In [ ]:
import pandas as pd
from pathlib import Path
from sklearn import pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
useTrain = pd.read_csv('../data/raw/train.csv')

X = useTrain.drop(columns=['SalePrice'])
y = useTrain['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Clase imputacion
class dataImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.modeBsmtExposure = None
        self.modeElectrical = None
        self.mappingLotFrontage = None
        self.globalMeanLotFrontage = None
        self.FEATURES_GROUP_LOTFRONTAGE = ['Street', 'LotShape', 'LandContour', 'Neighborhood']
    def fit(self, X, y=None):
        self.mappingLotFrontage = (X.groupby(self.FEATURES_GROUP_LOTFRONTAGE)['LotFrontage'].mean())
        self.globalMeanLotFrontage = X['LotFrontage'].mean()
        return self

    def transform(self, X):
        xDF = X.copy()

        #Imputer PoolQC
        xDF.loc[xDF['PoolArea'] == 0, 'PoolQC'] = 'NA'

        #Imputer LotFrontage
        IVLotFrontage = xDF.set_index(self.FEATURES_GROUP_LOTFRONTAGE).index.map(self.mappingLotFrontage)
        xDF['LotFrontage'] = xDF['LotFrontage'].fillna(IVLotFrontage)
        xDF['LotFrontage'] = xDF['LotFrontage'].fillna(self.globalMeanLotFrontage)

        return xDF



#imputacion de datos faltantes
def imputerPoolQC(rows):
    x = pd.DataFrame(rows, columns=['PoolQC', 'PoolArea'])
    x.loc[x['PoolArea'] == 0, 'PoolQC'] = 'NA'
    return x['PoolQC'].values

def imputerMiscFeature(rows):
    x = pd.DataFrame(rows, columns=['MiscFeature', 'MiscVal'])
    x.loc[x['MiscVal'] == 0, 'MiscFeature'] = 'NA'
    return x['MiscFeature'].values

def imputerAlley(rows):
    x = pd.DataFrame(rows, columns=['Alley'])
    x.loc[:, 'Alley'] = x['Alley'].fillna('NA')
    return x['Alley'].values

def imputerFence(rows):
    x = pd.DataFrame(rows, columns=['Fence'])
    x.loc[:, 'Fence'] = x['Fence'].fillna('NA')
    return x['Fence'].values

def imputerMasVnrArea(rows):
    x = pd.DataFrame(rows, columns=['MasVnrType', 'MasVnrArea'])
    x.loc[:,'MasVnrArea'] = x['MasVnrArea'].fillna(0)
    return x['MasVnrArea'].values

def imputerMasVnrType(rows):
    x = pd.DataFrame(rows, columns=['MasVnrType', 'MasVnrArea'])
    x.loc[:, 'MasVnrType'] = x['MasVnrType'].fillna('None')
    return x['MasVnrType'].values

def imputerFireplaceQu(rows):
    x = pd.DataFrame(rows, columns=['FireplaceQu', 'Fireplaces'])
    x.loc[x['Fireplaces'] == 0, 'FireplaceQu'] = 'NA'
    return x['FireplaceQu'].values

def imputerGarageFinish(rows):
    x = pd.DataFrame(rows, columns=['GarageFinish', 'GarageArea'])
    x.loc[x['GarageArea'] == 0, 'GarageFinish'] = 'NA'
    return x['GarageFinish'].values

def imputerGarageQual(rows):
    x = pd.DataFrame(rows, columns=['GarageQual', 'GarageArea'])
    x.loc[x['GarageArea'] == 0, 'GarageQual'] = 'NA'
    return x['GarageQual'].values

def imputerGarageType(rows):
    x = pd.DataFrame(rows, columns=['GarageType', 'GarageArea'])
    x.loc[x['GarageArea'] == 0, 'GarageType'] = 'NA'
    return x['GarageType'].values

def imputerGarageCond(rows):
    x = pd.DataFrame(rows, columns=['GarageCond', 'GarageArea'])
    x.loc[x['GarageArea'] == 0, 'GarageCond'] = 'NA'
    return x['GarageCond'].values

def imputerGarageYrBlt(rows):
    x = pd.DataFrame(rows, columns=['GarageYrBlt', 'GarageArea'])
    x.loc[x['GarageArea'] == 0, 'GarageYrBlt'] = 0
    return x['GarageYrBlt'].values

def imputerBsmtFinType1(rows):
    x = pd.DataFrame(rows, columns=['BsmtFinType1', 'TotalBsmtSF', 'BsmtUnfSF', 'BsmtFinSF1'])
    x.loc[x['TotalBsmtSF'] == 0, 'BsmtFinType1'] = 'NA'
    x.loc[(x['BsmtFinSF1'] != 0) & (x['BsmtUnfSF'] != 0), 'BsmtFinType1'] = 'Unf'
    return x['BsmtFinType1'].values

def imputerBsmtFinType2(rows):
    x = pd.DataFrame(rows, columns=['BsmtFinType2', 'TotalBsmtSF', 'BsmtUnfSF', 'BsmtFinSF2'])
    x.loc[x['TotalBsmtSF'] == 0, 'BsmtFinType2'] = 'NA'
    x.loc[(x['BsmtFinSF2'] != 0) & (x['BsmtUnfSF'] != 0), 'BsmtFinType2'] = 'Unf'
    return x['BsmtFinType2'].values

def imputerBsmtExposure(rows):
    x = pd.DataFrame(rows, columns=['BsmtExposure', 'TotalBsmtSF'])
    x.loc[x['TotalBsmtSF'] == 0, 'BsmtExposure'] = 'NA'
    return x['BsmtExposure'].values

def imputerBsmtCond(rows):
    x = pd.DataFrame(rows, columns=['BsmtCond', 'TotalBsmtSF'])
    x.loc[x['TotalBsmtSF'] == 0, 'BsmtCond'] = 'NA'
    return x['BsmtCond'].values

def imputerBsmtQual(rows):
    x = pd.DataFrame(rows, columns=['BsmtQual', 'TotalBsmtSF'])
    x.loc[x['TotalBsmtSF'] == 0, 'BsmtQual'] = 'NA'
    return x['BsmtQual'].values

def imputerLotFrontage(rows):
    x = pd.DataFrame(rows, columns=['Street', 'LotShape', 'LandContour', 'Neighborhood', 'LotFrontage'])
    x.loc[:, 'LotFrontage'] = x['LotFrontage'].fillna(x.groupby(['Street', 'LotShape', 'LandContour', 'Neighborhood'])['LotFrontage'].transform('mean'))
    return x['LotFrontage'].values

def imputerElectrical(rows):
    x = pd.DataFrame(rows, columns=['Electrical'])
    x.loc[:, 'Electrical'] = x['Electrical'].fillna(x['Electrical'].mode()[0])
    return x['Electrical'].values

def imputerBsmtExposure(rows):
    x = pd.DataFrame(rows, columns=['BsmtExposure', 'TotalBsmtSF'])
    x.loc[x['TotalBsmtSF'] == 0, 'BsmtExposure'] = x['BsmtExposure'].fillna(x['BsmtExposure'].mode()[0])
    return x['BsmtExposure'].values

In [ ]:
transformer_imputer = Pipeline([
    ('imputerPoolQC', FunctionTransformer(imputerPoolQC, validate=False)),
    ('imputerMiscFeature', FunctionTransformer(imputerMiscFeature, validate=False)),
    ('imputerAlley', FunctionTransformer(imputerAlley, validate=False)),
    ('imputerFence', FunctionTransformer(imputerFence, validate=False)),
    ('imputerMasVnrArea', FunctionTransformer(imputerMasVnrArea, validate=False)),
    ('imputerMasVnrType', FunctionTransformer(imputerMasVnrType, validate=False)),
    ('imputerFireplaceQu', FunctionTransformer(imputerFireplaceQu, validate=False)),
    ('imputerGarageFinish', FunctionTransformer(imputerGarageFinish, validate=False)),
    ('imputerGarageQual', FunctionTransformer(imputerGarageQual, validate=False)),
    ('imputerGarageType', FunctionTransformer(imputerGarageType, validate=False)),
    ('imputerGarageCond', FunctionTransformer(imputerGarageCond, validate=False)),
    ('imputerGarageYrBlt', FunctionTransformer(imputerGarageYrBlt, validate=False)),
    ('imputerBsmtFinType1', FunctionTransformer(imputerBsmtFinType1, validate=False)),
    ('imputerBsmtFinType2', FunctionTransformer(imputerBsmtFinType2, validate=False)),
    ('imputerBsmtExposure', FunctionTransformer(imputerBsmtExposure, validate=False)),
    ('imputerBsmtCond', FunctionTransformer(imputerBsmtCond, validate=False)),
    ('imputerBsmtQual', FunctionTransformer(imputerBsmtQual, validate=False)),
    ('imputerLotFrontage', FunctionTransformer(imputerLotFrontage, validate=False)),
    ('imputerElectrical', FunctionTransformer(imputerElectrical, validate=False)),
    ('imputerBsmtExposure', FunctionTransformer(imputerBsmtExposure, validate=False))

])



X_train['PoolQC'] = transformer_imputer.named_steps['imputerPoolQC'].transform(X_train[['PoolQC', 'PoolArea']])
X_train['MiscFeature'] = transformer_imputer.named_steps['imputerMiscFeature'].transform(X_train[['MiscFeature', 'MiscVal']])
X_train['Alley'] = transformer_imputer.named_steps['imputerAlley'].transform(X_train[['Alley']])
X_train['Fence'] = transformer_imputer.named_steps['imputerFence'].transform(X_train[['Fence']])
X_train['MasVnrArea'] = transformer_imputer.named_steps['imputerMasVnrArea'].transform(X_train[['MasVnrType', 'MasVnrArea']])
X_train['MasVnrType'] = transformer_imputer.named_steps['imputerMasVnrType'].transform(X_train[['MasVnrType', 'MasVnrArea']])
X_train['FireplaceQu'] = transformer_imputer.named_steps['imputerFireplaceQu'].transform(X_train[['FireplaceQu', 'Fireplaces']])
X_train['GarageFinish'] = transformer_imputer.named_steps['imputerGarageFinish'].transform(X_train[['GarageFinish', 'GarageArea']])
X_train['GarageQual'] = transformer_imputer.named_steps['imputerGarageQual'].transform(X_train[['GarageQual', 'GarageArea']])
X_train['GarageType'] = transformer_imputer.named_steps['imputerGarageType'].transform(X_train[['GarageType', 'GarageArea']])
X_train['GarageCond'] = transformer_imputer.named_steps['imputerGarageCond'].transform(X_train[['GarageCond', 'GarageArea']])
X_train['GarageYrBlt'] = transformer_imputer.named_steps['imputerGarageYrBlt'].transform(X_train[['GarageYrBlt', 'GarageArea']])
X_train['BsmtFinType1'] = transformer_imputer.named_steps['imputerBsmtFinType1'].transform(X_train[['BsmtFinType1', 'TotalBsmtSF', 'BsmtUnfSF', 'BsmtFinSF1']])
X_train['BsmtFinType2'] = transformer_imputer.named_steps['imputerBsmtFinType2'].transform(X_train[['BsmtFinType2', 'TotalBsmtSF', 'BsmtUnfSF', 'BsmtFinSF2']])
X_train['BsmtExposure'] = transformer_imputer.named_steps['imputerBsmtExposure'].transform(X_train[['BsmtExposure', 'TotalBsmtSF']])
X_train['BsmtCond'] = transformer_imputer.named_steps['imputerBsmtCond'].transform(X_train[['BsmtCond', 'TotalBsmtSF']])
X_train['BsmtQual'] = transformer_imputer.named_steps['imputerBsmtQual'].transform(X_train[['BsmtQual', 'TotalBsmtSF']])
X_train['LotFrontage'] = transformer_imputer.named_steps['imputerLotFrontage'].transform(X_train[['Street', 'LotShape', 'LandContour', 'Neighborhood', 'LotFrontage']])
X_train['Electrical'] = transformer_imputer.named_steps['imputerElectrical'].transform(X_train[['Electrical']])
X_train['BsmtExposure'] = transformer_imputer.named_steps['imputerBsmtExposure'].transform(X_train[['BsmtExposure', 'TotalBsmtSF']])

X_train = pd.DataFrame(X_train)
condition = X_train['PoolQC'].isnull() | X_train['MiscFeature'].isnull() | X_train['Alley'].isnull() | X_train['Fence'].isnull() | X_train['MasVnrArea'].isnull() | X_train['MasVnrType'].isnull() | X_train['FireplaceQu'].isnull() | X_train['GarageFinish'].isnull() | X_train['GarageQual'].isnull() | X_train['GarageType'].isnull() | X_train['GarageCond'].isnull() | X_train['GarageYrBlt'].isnull() | X_train['BsmtFinType1'].isnull() | X_train['BsmtFinType2'].isnull() | X_train['BsmtExposure'].isnull() | X_train['BsmtCond'].isnull() | X_train['BsmtQual'].isnull() | X_train['LotFrontage'].isnull() | X_train['Electrical'].isnull() | X_train['BsmtExposure'].isnull()
X_train[condition]
